In [1]:
#Calling and running the embeddings.py deliberately keeping it separate to keep the codebase clean
from embeddings import run
df, tfidf_df, sbert_df, e5_df = run()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aamin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Embeddings check
  TF-IDF    150 docs x 100 dims
  SBERT     150 docs x 384 dims
  E5        150 docs x 768 dims

  Companies: 50
  Years:     [2019, 2021, 2023]



**some comments** for future ref.

Before we can cluster companies into themes, we need to convert the raw 10-K text into numbers. This step does that using three different embedding methods, which we will compare in the clustering step to see which produces the most meaningful themes. We first clean the text by stripping numbers, punctuation and extra whitespace, since we care about the language a company uses to describe its business rather than financial figures.

- TF-IDF is the simple baseline - it scores words by how distinctive they are to a given document compared to the rest. We compress this down to 100 numbers per document. It is fast and transparent but has no real understanding of meaning, it just counts words.

- Sentence-BERT goes further by actually understanding language, so two companies describing the same business in different words still end up close together. Each document becomes a vector of 384 numbers. E5 is the strongest of the three and was built specifically for similarity tasks like this one, outputting 768 numbers per document.

- The dimensions (100, 384, 768) just refer to the length of the number list representing each document. The individual numbers are not interpretable on their own, what matters is that similar companies end up with similar vectors, which the clustering step in Part 3 uses to find groups.

Output: 150 documents x 3 embedding methods (TF-IDF 100 dims, SBERT 384 dims, E5 768 dims).

#### A quick note on dimensions


Current output:
- TF-IDF:  150 documents x 100 dimensions
- SBERT:   150 documents x 384 dimensions
- E5:      150 documents x 768 dimensions

on the 100 or 384 or 768 dimensions,(length of the number list that represents each document, which is not really interpretable on their own). What matters is that documents about similar businesses end up with similar number lists, meaning they sit near each other in space. The clustering step (the next step) takes advantage of that to find groups which become the themes.


#### Where we are

    - we pulled data - 150 filings pulled from EDGAR (50 companies, 3 years each)
    - converted filings to embedding vectors for the three methods above
    - clustering those vectors to discover themes is next.

In [13]:
import importlib, clustering; importlib.reload(clustering); from clustering import run

In [14]:
from clustering import run

# constrained - strips boilerplate, cleaner themes
results_c, summary_c = run(constrained=True, force=True)

# free - raw embeddings, use for comparison in writeup
results_f, summary_f = run(constrained=False, force=True)

C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n


Clustering check  [constrained]
  tfidf  2019  ->  12 themes,  7 noise
  tfidf  2021  ->  15 themes,  9 noise
  tfidf  2023  ->  14 themes,  7 noise
  sbert  2019  ->  16 themes,  7 noise
  sbert  2021  ->  17 themes,  4 noise
  sbert  2023  ->  15 themes,  7 noise
  e5     2019  ->  16 themes,  5 noise
  e5     2021  ->  15 themes,  7 noise
  e5     2023  ->  16 themes,  8 noise



C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\aamin\anaconda3\envs\mfe_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n


Clustering check  [free]
  tfidf  2019  ->  15 themes,  14 noise
  tfidf  2021  ->  15 themes,  4 noise
  tfidf  2023  ->  15 themes,  5 noise
  sbert  2019  ->  16 themes,  7 noise
  sbert  2021  ->  17 themes,  4 noise
  sbert  2023  ->  15 themes,  7 noise
  e5     2019  ->  16 themes,  5 noise
  e5     2021  ->  15 themes,  7 noise
  e5     2023  ->  16 themes,  8 noise



In [15]:
exec(open("print_themes.py").read())               # constrained (default)


Theme buckets  [constrained]
Loaded 150 filings


  TFIDF 2019  |  12 themes  |  7 noise  |  [constrained]

  noise : AAPL, ACN, GOOGL, GS, META, TMO, TXN

  theme 0 (5 companies)
  companies : BAC, BRK-B, JPM, SPGI, WFC
  top words : insurance, bank, banks, ratings, reinsurance, loan, supervision, fdic, banking, deposits

  theme 1 (6 companies)
  companies : ABBV, ABT, JNJ, LLY, MRK, UNH
  top words : products, care, treatment, health, health care, abbott, patent, approved, pharmaceutical, fda

  theme 2 (2 companies)
  companies : CVX, XOM
  top words : oil, gas, natural gas, crude, crude oil, production, natural, wells, reserves, thousand

  theme 3 (5 companies)
  companies : AMZN, COST, DIS, NFLX, WMT
  top words : walmart, television, streaming, programming, members, entertainment, content, merchandise, membership, disney

  theme 4 (2 companies)
  companies : GE, MCD
  top words : catherine, proxy statement, february, pursuant requirements, duly, signed, miller, jamie, proxy, 


  theme 3 (4 companies)
  companies : BAC, INTU, LIN, WFC
  top words : gases, bank, banks, fdic, banking, holding, customers, federal, services, products

  theme 4 (2 companies)
  companies : COST, GOOGL
  top words : google, merchandise, warehouses, members, ads, advertising, membership, memberships, advertisers, products

  theme 5 (2 companies)
  companies : DIS, NVDA
  top words : television, programming, gpus, disney, ai, gaming, gpu, computing, networks, entertainment

  theme 6 (3 companies)
  companies : QCOM, RTX, SPGI
  top words : ratings, aircraft, systems, revenue, aerospace, carrier, credit, trademarks, market, engine

  theme 7 (2 companies)
  companies : AMZN, WMT
  top words : walmart, amazon, stores, april, certain, ceo, served, publishers, fulfillment, factors

  theme 8 (2 companies)
  companies : PM, V
  top words : tobacco, heated, trade names, expectations regarding, names trademarks, trademarks, products, international, units, region

  theme 9 (3 companies)



  theme 2 (2 companies)
  companies : GS, JPM
  top words : loan, regulation, banking global, human capital, global markets, wealth, assets, cautionary statement, deposits, lending

  theme 3 (2 companies)
  companies : PG, SPGI
  top words : ratings, note, products, credit, revenue, stores, customers, consolidated, intelligence, credit ratings

  theme 4 (6 companies)
  companies : AMD, AVGO, COST, KO, NEE, PEP
  top words : graphics, data, performance, words, reform act, meaning, certain, reform, anticipate, words similar

  theme 5 (3 companies)
  companies : LLY, PM, TMO
  top words : products, tobacco, patent, instruments, treatment, clinical, laboratory, heated, used, pharmaceutical

  theme 6 (2 companies)
  companies : BAC, WFC
  top words : bank, banks, fdic, banking, holding, depository, federal, parent, deposit insurance, deposit

  theme 7 (2 companies)
  companies : CVX, XOM
  top words : oil, gas, natural gas, crude, crude oil, production, natural, wells, reserves, thous

**Comments**

With the embeddings in place, we now group companies into themes by clustering their vectors. The idea is that companies whose 10-K language sits close together in embedding space are likely operating in the same theme, even if no one told the model what the themes are. We run clustering independently for each embedding method and each year, giving us 9 sets of results to compare.

HDBSCAN is the clustering algorithm. We chose this over simpler methods like KMeans because it finds natural groupings without requiring us to specify the number of themes upfront. It also marks companies that do not clearly belong anywhere as noise rather than forcing them into a group, which is more honest.
UMAP is used to reduce the embedding dimensions before clustering. This is a standard step that makes the clusters more stable and also gives us 2D coordinates we can use for visualization later.
Soft affinity scores are the key output. Rather than a hard yes/no assignment, each company gets a score between 0 and 1 per theme. A company like MSFT might score 0.85 on one theme and 0.40 on another, which is more useful than a single label and sets up the temporal analysis nicely.

Output:
we see the themes discovered per method:

    - TF-IDF: 4-7 themes per year, consistently finds the financials cluster (BAC, BRK-B, JPM, WFC) across all years
    - SBERT: 2-7 themes per year, broader clusters, some large catch-all groups in earlier years
    - E5: 3-6 themes per year, tightest clusters, cleanest theme separation in 2023 (healthcare, energy clearly isolated)


pulled 150 filings from EDGAR (50 companies, 3 years each)
converted filings to embedding vectors using TF-IDF, SBERT and E5
clustered those vectors into themes with soft affinity scores per company
next step is validation - checking how well our discovered themes align with known thematic ETF baskets

# Part 3 - Clustering

With the embeddings in place, we now group companies into themes by clustering their vectors. The idea is that companies whose 10-K language sits close together in embedding space are likely operating in the same theme, even if no one told the model what the themes are. We run clustering independently for each embedding method and each year, giving us 9 sets of results to compare.

HDBSCAN is the clustering algorithm. We chose this over simpler methods like KMeans because it finds natural groupings without requiring us to specify the number of themes upfront. It also marks companies that do not clearly belong anywhere as noise rather than forcing them into a group, which is more honest. UMAP is used to reduce the embedding dimensions before clustering, which makes the clusters more stable and gives us 2D coordinates for visualization later.

Soft affinity scores are the key output. Rather than a hard yes/no assignment, each company gets a score between 0 and 1 per theme. A company like MSFT might score 0.85 on one theme and 0.40 on another, which is more useful than a single label and sets up the temporal analysis nicely.

We run clustering in two modes: constrained strips out 10-K boilerplate (words like "item", "statements", "forward looking") that appear in every filing and carry no thematic signal. Free uses the raw embeddings. Both are saved separately so we can compare them in the writeup.

Output - themes discovered per method after applying the constrained mode:

- TF-IDF: 12-15 themes per year, granular clusters, consistently isolates financials (BAC, JPM, WFC), healthcare (ABBV, JNJ, MRK, UNH) and energy (CVX, LIN, XOM) across all years
- SBERT: 15-17 themes per year, some unexpected pairings but broadly coherent, healthcare and energy stable
- E5: 15-16 themes per year, cleanest separation, by 2023 clearly identifies AI/cloud (CRM, GOOGL, ORCL), payments (MA, V), and pharma as distinct themes



As the next step we need to run validation to check how well the discovered themes align with known thematic ETF baskets. (not a perfect science here as the ETFs take long to launch, and are imperfect in design anyway - for example due to cocerns like having a min number of constituents to ensure liquidity therefore diluting the thematic purity of the ETF for more 'investability')

In [17]:
from validation import run
overlap_results, lead_results, summary = run()


Validation Results

Part A - ETF Overlap (best match per cluster, F1 > 0.2 shown)
------------------------------------------------------------

  TFIDF 2019
    cluster 0    FINX_fintech               F1=0.25  P=0.20  R=0.33  overlap: SPGI
    cluster 1    ARKG_genomics              F1=0.25  P=0.17  R=0.50  overlap: ABBV
    cluster 3    ARKK_innovation            F1=0.25  P=0.20  R=0.33  overlap: AMZN
    cluster 5    FINX_fintech               F1=0.40  P=0.29  R=0.67  overlap: MA, V
    cluster 6    IGV_software               F1=0.22  P=0.33  R=0.17  overlap: INTU
    cluster 8    IGV_software               F1=0.80  P=1.00  R=0.67  overlap: ADBE, CRM, MSFT, ORCL
    cluster 10   BOTZ_ai_robotics           F1=0.80  P=0.67  R=1.00  overlap: IBM, NVDA
    cluster 11   ARKK_innovation            F1=0.40  P=0.50  R=0.33  overlap: TSLA

  TFIDF 2021
    cluster 0    FINX_fintech               F1=0.40  P=0.33  R=0.50  overlap: MA, V
    cluster 3    HACK_cybersecurity         F1=0.40  P=0.

In [18]:
overlap_results, lead_results, summary = run(fetch_live=True)



Validation Results

Part A - ETF Overlap (best match per cluster, F1 > 0.2 shown)
------------------------------------------------------------

  TFIDF 2019
    cluster 0    FINX_fintech               F1=0.25  P=0.20  R=0.33  overlap: SPGI
    cluster 1    ARKG_genomics              F1=0.25  P=0.17  R=0.50  overlap: ABBV
    cluster 3    ARKK_innovation            F1=0.25  P=0.20  R=0.33  overlap: AMZN
    cluster 5    FINX_fintech               F1=0.40  P=0.29  R=0.67  overlap: MA, V
    cluster 6    IGV_software               F1=0.22  P=0.33  R=0.17  overlap: INTU
    cluster 8    IGV_software               F1=0.80  P=1.00  R=0.67  overlap: ADBE, CRM, MSFT, ORCL
    cluster 10   BOTZ_ai_robotics           F1=0.80  P=0.67  R=1.00  overlap: IBM, NVDA
    cluster 11   ARKK_innovation            F1=0.40  P=0.50  R=0.33  overlap: TSLA

  TFIDF 2021
    cluster 0    FINX_fintech               F1=0.40  P=0.33  R=0.50  overlap: MA, V
    cluster 3    HACK_cybersecurity         F1=0.40  P=0.

# ********* Items to address next time I pick this up:  ********* 
1. ETF launch lag caveat
ETFs don't launch the moment a theme emerges — regulatory filing, seed capital, index construction and admin delays mean there's typically a 12-24 month gap between a theme becoming investable and an ETF actually launching. This means our measured lead times are likely understated. Worth noting in the writeup as a conservative bias — if anything, our early detection metric is more impressive than the numbers suggest.
2. The data leakage / data mining concern — this is the critical one
The honest question is: when we run the 2019 analysis using only 2019 filings, are we genuinely detecting an emerging AI theme, or are we just pattern-matching on things we already know in hindsight? We need to explicitly confirm and document that:

The 2019 clustering used only 2019 10-K text, with no look-ahead
The ETF validation labels were applied after the clustering, not used as inputs
The model had no knowledge of future ETF composition when it ran

This is the most important methodological point for academic credibility and for recruiting conversations. We should add a dedicated note in the markdown and a short paragraph in the writeup that addresses this directly.
3. Visualization
Still to build — UMAP plots of clusters per year, showing how companies move between themes across 2019/2021/2023. That's the most compelling visual for the paper and for pitching the project.

In [20]:
import subprocess

commands = [
    "git init",
    "git remote add origin https://github.com/AA-mini/ThemeDrift.git",  # skip if already added
    "git add .",
    'git commit -m "add ThemeDrift pipeline - data pull, embeddings, clustering, validation"',
    "git branch -M main",
    "git push -u origin main",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)

$ git init
Reinitialized existing Git repository in C:/Users/aamin/NLP Final Project/.git/

$ git remote add origin https://github.com/AA-mini/ThemeDrift.git
error: remote origin already exists.

$ git add .
$ git commit -m "add ThemeDrift pipeline - data pull, embeddings, clustering, validation"
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean

$ git branch -M main
$ git push -u origin main
branch 'main' set up to track 'origin/main'.

Everything up-to-date

